In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

# Caminho de saída dos HTMLs — ajusta conforme o teu projeto
OUTPUT_DIR = "./assets/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset = pd.read_csv("dataset_limpo_medianag.csv")

A guardar em: C:\Users\marco\OneDrive\Ambiente de Trabalho\dashoard_UPT\DataSet_GroupProject\Public\assets\
Pasta existe? True


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\marco\\OneDrive\\Ambiente de Trabalho\\dashoard_UPT\\DataSet_GroupProject\\Public\\DATASET\\dataset_limpo_medianag.csv'

In [8]:
#G1: Seniority Distribution (Donut)
df_g2 = dataset.copy()
df_g2["YearsCodePro"] = pd.to_numeric(
    df_g2["YearsCodePro"].replace({"Less than 1 year": 0, "More than 50 years": 51}),
    errors="coerce")

def seniority(y):
    if pd.isna(y):  return None
    if y < 2:       return "Junior"
    if y < 6:       return "Mid"
    if y < 12:      return "Senior"
    return "Expert"

df_g2["Seniority"] = df_g2["YearsCodePro"].apply(seniority)
df_g2 = df_g2[df_g2["Seniority"].notna()]

ordem     = ["Junior", "Mid", "Senior", "Expert"]
cores_sen = ["#1e88e5", "#14b8a6", "#fbbf24", "#fb7f3f"]
sen_counts = df_g2["Seniority"].value_counts().reindex(ordem).fillna(0).reset_index()
sen_counts.columns = ["Seniority", "Count"]

fig = go.Figure(go.Pie(
    labels=sen_counts["Seniority"],
    values=sen_counts["Count"],
    hole=0.5,
    marker=dict(colors=cores_sen, line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12, color="white")
))
fig.update_layout(
    title="Seniority Distribution in the Market",
    title_font_size=16,
    margin=dict(l=20, r=20, t=60, b=20),
    showlegend=True
)
fig.add_annotation(text="Seniority", x=0.5, y=0.5,
                   font=dict(size=14, color="black"), showarrow=False)

fig.write_html(OUTPUT_DIR + "g1_seniority.html")
fig.show(renderer="browser")

In [9]:
#G2: Age Distribution (Barras horizontais)
ordem_age   = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
geracao_map = {"<18":"Gen Alpha", "18-24":"Gen Z", "25-34":"Millennials",
               "35-44":"Gen X", "45-54":"Boomers", "55-64":"Boomers", "65+":"Silent"}
cores_ger   = {"Gen Alpha":"#7c78d1", "Gen Z":"#1e88e5", "Millennials":"#14b8a6",
               "Gen X":"#fbbf24", "Boomers":"#fb7f3f", "Silent":"#64748b"}

age_counts = (dataset["Age"].dropna()
              .replace("Sem dado", None).dropna()
              .value_counts().reindex(ordem_age).fillna(0).reset_index())
age_counts.columns    = ["Age", "Count"]
age_counts["Percent"] = (age_counts["Count"] / len(dataset)) * 100
age_counts["Geração"] = age_counts["Age"].map(geracao_map)
age_counts["Color"]   = age_counts["Geração"].map(cores_ger)

fig = px.bar(
    age_counts, x="Percent", y="Age", orientation="h",
    color="Geração",
    color_discrete_map=cores_ger,
    text=age_counts.apply(lambda r: f'{r["Percent"]:.1f}% — {r["Geração"]}', axis=1),
    title="Age Distribution in the Programming Market",
    labels={"Percent": "% of Respondents", "Age": "Age Group"}
)
fig.update_traces(textposition="outside")
fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(ordem_age)))
fig.update_layout(
    title_font_size=16,
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(showgrid=False),
    plot_bgcolor="white"
)

fig.write_html(OUTPUT_DIR + "g2_age.html")
fig.show(renderer="browser")

In [10]:
#G3: Salary by Developer Type (Boxplot)
df_g1 = dataset[dataset["ConvertedCompYearly"] != 65000]
df_g1 = df_g1[["DevType", "ConvertedCompYearly"]].dropna()
df_g1 = df_g1.assign(DevType=df_g1["DevType"].str.split(";")).explode("DevType")
df_g1["DevType"] = df_g1["DevType"].str.strip().replace("Sem dado", None)
df_g1 = df_g1.dropna()

top_devtypes = df_g1["DevType"].value_counts().head(10).index
df_g1 = df_g1[df_g1["DevType"].isin(top_devtypes)]
cap   = df_g1["ConvertedCompYearly"].quantile(0.95)
df_g1 = df_g1[df_g1["ConvertedCompYearly"] <= cap]

fig = px.box(
    df_g1, x="ConvertedCompYearly", y="DevType",
    orientation="h",
    color="DevType",
    title="Salary Distribution by Developer Type",
    labels={"ConvertedCompYearly": "Annual Salary (USD)", "DevType": "Developer Type"}
)
fig.update_layout(
    title_font_size=16,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=20),
    plot_bgcolor="white"
)

fig.write_html(OUTPUT_DIR + "g3_salary_devtype.html")
fig.show(renderer="browser")

In [11]:
#G4: Median Salary by Experience (Line)
df_g3 = dataset[dataset["ConvertedCompYearly"] != 65000]
df_g3 = df_g3[["YearsCodePro", "ConvertedCompYearly"]].dropna()
df_g3["YearsCodePro"] = pd.to_numeric(
    df_g3["YearsCodePro"].replace({"Less than 1 year": 0, "More than 50 years": 51}),
    errors="coerce")
df_g3 = df_g3.dropna()
cap   = df_g3["ConvertedCompYearly"].quantile(0.95)
df_g3 = df_g3[df_g3["ConvertedCompYearly"] <= cap]

bins   = [0, 2, 5, 10, 15, 20, 30, 51]
labels = ["0-2", "3-5", "6-10", "11-15", "16-20", "21-30", "30+"]
df_g3["GrupoExp"] = pd.cut(df_g3["YearsCodePro"], bins=bins, labels=labels, right=True)
mediana_grupo = df_g3.groupby("GrupoExp", observed=True)["ConvertedCompYearly"].median().reset_index()
mediana_grupo.columns = ["GrupoExp", "MedianSalary"]
mediana_grupo["MedianSalary_K"] = mediana_grupo["MedianSalary"] / 1000

fig = px.line(
    mediana_grupo, x="GrupoExp", y="MedianSalary_K",
    markers=True,
    title="Median Salary by Years of Experience",
    labels={"GrupoExp": "Years of Professional Experience", "MedianSalary_K": "Median Salary (USD K)"},
    text=mediana_grupo["MedianSalary_K"].apply(lambda v: f"${v:.0f}K")
)
fig.update_traces(
    line=dict(color="#1e88e5", width=2.5),
    marker=dict(color="#e53935", size=9),
    textposition="top center"
)
fig.update_layout(
    title_font_size=16,
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g4_salary_exp.html")
fig.show(renderer="browser")

In [12]:
#G5: Most Used Tools (Barras horizontais)
tools_counts = (
    dataset["ToolsTechHaveWorkedWith"]
    .dropna().str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(15).reset_index()
)
tools_counts.columns    = ["Tool", "Count"]
tools_counts["Percent"] = (tools_counts["Count"] / len(dataset)) * 100

fig = px.bar(
    tools_counts.sort_values("Percent"),
    x="Percent", y="Tool", orientation="h",
    text=tools_counts.sort_values("Percent")["Percent"].apply(lambda v: f"{v:.1f}%"),
    title="Most Used Development Tools",
    labels={"Percent": "% of Respondents", "Tool": ""},
    color_discrete_sequence=["#14b8a6"]
)
fig.update_traces(textposition="outside", marker_color="#14b8a6")
fig.update_layout(
    title_font_size=16,
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g5_tools.html")
fig.show(renderer="browser")

In [13]:
#G6: Remote Work (Donut)
remote_counts = (
    dataset["RemoteWork"].dropna()
    .replace("Sem dado", None).dropna()
    .value_counts().reset_index()
)
remote_counts.columns    = ["RemoteWork", "Count"]
remote_counts["Percent"] = (remote_counts["Count"] / remote_counts["Count"].sum()) * 100

fig = go.Figure(go.Pie(
    labels=remote_counts["RemoteWork"],
    values=remote_counts["Count"],
    hole=0.5,
    marker=dict(colors=["#1e88e5", "#14b8a6", "#fbbf24"],
                line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12)
))
fig.add_annotation(text="Remote<br>Work", x=0.5, y=0.5,
                   font=dict(size=13, color="black"), showarrow=False)
fig.update_layout(
    title="Remote, Hybrid or In-Person Work Preference",
    title_font_size=16,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g6_remote.html")
fig.show(renderer="browser")

In [14]:
#G7: Job Satisfaction Factors (Bar horizontal)
jobsat_cols = {
    "JobSatPoints_1":  "Industry",
    "JobSatPoints_4":  "Management",
    "JobSatPoints_5":  "Languages",
    "JobSatPoints_6":  "Compensation",
    "JobSatPoints_7":  "Flexibility",
    "JobSatPoints_8":  "Environment",
    "JobSatPoints_9":  "Diversity",
    "JobSatPoints_10": "Mission/Values",
    "JobSatPoints_11": "Learning",
}

medias = {}
for col, label in jobsat_cols.items():
    if col in dataset.columns:
        valores = pd.to_numeric(dataset[col], errors="coerce").dropna()
        medias[label] = valores.mean()

medias_df = pd.Series(medias).sort_values(ascending=True).reset_index()
medias_df.columns = ["Fator", "Media"]

fig = px.bar(
    medias_df, x="Media", y="Fator", orientation="h",
    text=medias_df["Media"].apply(lambda v: f"{v:.1f} pts"),
    title="What Do Developers Value Most at Work?",
    labels={"Media": "Average Score", "Fator": ""},
    color_discrete_sequence=["#1e3a5f"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    title_font_size=16,
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g7_jobsat.html")
fig.show(renderer="browser")

In [15]:
# G8: Industries (Treemap)
industry_counts = (
    dataset["Industry"].dropna()
    .replace("Sem dado", None).dropna()
    .value_counts().head(12).reset_index()
)
industry_counts.columns    = ["Industry", "Count"]
industry_counts["Percent"] = (industry_counts["Count"] / len(dataset)) * 100
industry_counts["Label"]   = industry_counts.apply(
    lambda r: f'{r["Industry"]}<br>{r["Percent"]:.1f}%', axis=1)

fig = px.treemap(
    industry_counts,
    path=["Industry"], values="Count",
    color="Count",
    color_continuous_scale=["#1e88e5","#14b8a6","#fbbf24","#fb7f3f","#7c78d1"],
    title="Industries Where Developers Currently Work",
    custom_data=["Percent"]
)
fig.update_traces(
    texttemplate="%{label}<br>%{customdata[0]:.1f}%",
    textfont=dict(size=13)
)
fig.update_layout(
    title_font_size=16,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g8_industries.html")
fig.show(renderer="browser")

In [16]:
#G9: Mapa Mundial (Choropleth)
nome_map = {
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Viet Nam": "Vietnam", "Russian Federation": "Russia",
    "Republic of Korea": "South Korea", "Republic of Moldova": "Moldova",
    "Iran, Islamic Republic of...": "Iran", "Venezuela, Bolivarian Republic of...": "Venezuela",
    "Hong Kong (S.A.R.)": "Hong Kong", "Sem dado": None, "Nomadic": None,
}

df_mapa = dataset[dataset["ConvertedCompYearly"] != 65000]
df_mapa = df_mapa[["Country", "ConvertedCompYearly"]].dropna()
df_mapa["Country"] = df_mapa["Country"].replace(nome_map)
df_mapa = df_mapa[df_mapa["Country"].notna()]

paises_validos = df_mapa["Country"].value_counts()
paises_validos = paises_validos[paises_validos >= 50].index
df_mapa = df_mapa[df_mapa["Country"].isin(paises_validos)]

mediana_pais = df_mapa.groupby("Country")["ConvertedCompYearly"].median().reset_index()
mediana_pais.columns      = ["Country", "Mediana"]
mediana_pais["Mediana_K"] = (mediana_pais["Mediana"] / 1000).round(1)

fig = px.choropleth(
    mediana_pais, locations="Country", locationmode="country names",
    color="Mediana_K", color_continuous_scale="Blues",
    title="Median Salary by Country (countries with ≥ 50 respondents)",
    labels={"Mediana_K": "Median Salary (USD K)"},
    hover_name="Country"
)
fig.update_layout(
    title_font_size=16, title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.write_html(OUTPUT_DIR + "g9_map.html")
fig.show(renderer="browser")

C:\Users\marco\AppData\Local\Temp\ipykernel_17460\1460138901.py:23: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(
